<a href="https://colab.research.google.com/github/mayait/CursoAnalisisDatos_IA_2026/blob/main/sitio/labs/lab_16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Laboratorio 16 · La defensa

Quince semanas para llegar aquí. La semana pasada auditaste tu modelo y escribiste sus límites; hoy no
se analiza nada nuevo. **Hoy se prepara la entrega**: diez minutos delante de un panel que no leyó tu
cuaderno, más diez de preguntas de gente que va a buscar exactamente el punto flojo que tú ya conoces.

Este cuaderno es una caja de herramientas, no una clase. Tiene el gráfico de defensa comparado con el
de exploración sobre el mismo dato, el guion de diez minutos con la conclusión en el primer minuto, un
**verificador automático de tu propio cuaderno** que te dice qué te falta antes de entregarlo, el banco
de preguntas del panel con la evidencia que hay que tener abierta para cada una, y la rúbrica con la
que te van a calificar. Se usa con el proyecto del grupo al lado, no después.

> **Hoy haces** · Conviertes un gráfico de exploración en uno de defensa y anotas las siete diferencias
> (90 min). Armas el guion de diez minutos con la pirámide de Minto y lo cronometras por secciones.
> Pasas el verificador sobre el cuaderno de tu grupo y arreglas lo que salga pendiente. Repasas el banco
> de preguntas del panel y localizas, para cada una, la celda exacta de tu cuaderno que la contesta.
> Cierras leyendo la rúbrica y puntuándote tú mismo antes de que lo haga el panel.
>
> **Entrega** · Proyecto final: recomendación de una página más cuaderno reproducible, defendido ante
> panel. Se entrega además la bitácora de prompts completa y la evaluación 360 entre pares.
> Este cuaderno se sube con el verificador ejecutado sobre el archivo del grupo y con la salida visible.
> Nombre de archivo: `lab_16_apellido.ipynb`.

In [ ]:
# --- Setup del entorno ---
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# Los datos de Comercial Andina viven en sitio/datos/
REPO = "https://github.com/mayait/CursoAnalisisDatos_IA_2026.git"
COPIA = Path("/content/CursoAnalisisDatos_IA_2026")
CANDIDATOS = [Path("../datos"), Path("datos"), Path("sitio/datos"),
              COPIA / "sitio" / "datos"]
DATOS = next((p for p in CANDIDATOS if p.exists()), None)
if DATOS is None:
    # En Colab el cuaderno llega solo: se trae el repositorio una sola vez.
    import subprocess
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(COPIA)], check=True)
    DATOS = COPIA / "sitio" / "datos"

print("Setup completo ✓")
print(f"pandas {pd.__version__} · datos en {DATOS.resolve()}")

## 1. El mismo dato, dos gráficos, dos oficios

Un gráfico de exploración lo miras tú, en tu pantalla, a treinta centímetros, y lo miras **para
descubrir**: quieres todas las series, todos los puntos y el eje completo. Un gráfico de defensa lo
mira un panel a seis metros, proyectado, durante **once segundos**, y lo mira **para entender una
conclusión que tú ya sacaste**.

Son dos productos distintos. El error de siempre es proyectar el primero.

Empezamos calculando el dato: ventas mensuales por ciudad de Comercial Andina y el crecimiento de los
últimos doce meses contra los doce anteriores.

In [ ]:
ventas = pd.read_csv(DATOS / "ventas_limpias.csv", parse_dates=["fecha"])
sucursales = pd.read_csv(DATOS / "sucursales.csv")

v = ventas.merge(sucursales[["sucursal_id", "ciudad"]], on="sucursal_id",
                 how="left", validate="m:1")
v["monto"] = v["cantidad"] * v["precio_unitario"] * (1 - v["descuento"])
v["mes"] = v["fecha"].values.astype("datetime64[M]")

serie = v.groupby(["mes", "ciudad"])["monto"].sum().unstack()
serie = serie[serie.index < "2026-07-01"]          # julio no está cerrado (lab 12)
suave = serie.rolling(3, min_periods=1).mean()      # media móvil de tres meses

ultimos12 = serie.iloc[-12:].sum()
previos12 = serie.iloc[-24:-12].sum()
crecimiento = ((ultimos12 / previos12 - 1) * 100).sort_values()

print(f"{len(serie)} meses · de {serie.index.min():%m-%Y} a {serie.index.max():%m-%Y}\n")
print("Participación en la facturación total:")
print((serie.sum() / serie.sum().sum() * 100).sort_values(ascending=False)
      .to_string(float_format=lambda x: f"{x:5.2f} %"))
print(f"\nCrecimiento de los últimos 12 meses contra los 12 anteriores "
      f"(nacional {(ultimos12.sum() / previos12.sum() - 1) * 100:.2f} %):")
print(crecimiento.to_string(float_format=lambda x: f"{x:+6.2f} %"))
print(f"\nLa conclusión del gráfico: {crecimiento.index[-1]} crece al "
      f"{crecimiento.iloc[-1]:.1f} % y {crecimiento.index[0]}, que es la segunda plaza del país "
      f"({serie.sum()[crecimiento.index[0]] / serie.sum().sum() * 100:.1f} % de la facturación), "
      f"crece al {crecimiento.iloc[0]:.1f} %.")

In [ ]:
PROTAGONISTA, REZAGADA = crecimiento.index[-1], crecimiento.index[0]

fig, axes = plt.subplots(1, 2, figsize=(15, 4.6))

# ── A · gráfico de exploración: todo, sin jerarquía ──────────────────────
for ciudad in serie.columns:
    axes[0].plot(serie.index, serie[ciudad], linewidth=1, marker="o", markersize=3,
                 label=ciudad)
axes[0].legend(fontsize=6, ncol=2)
axes[0].set_title("monto por mes y ciudad", fontsize=8)
axes[0].set_xlabel("mes", fontsize=7)
axes[0].set_ylabel("monto", fontsize=7)
axes[0].tick_params(labelsize=6)

# ── B · gráfico de defensa: una idea, dos series, texto grande ───────────
for ciudad in suave.columns:
    if ciudad not in (PROTAGONISTA, REZAGADA):
        axes[1].plot(suave.index, suave[ciudad] / 1000, color="#D9D9D9", linewidth=1.8)
for ciudad, color in [(PROTAGONISTA, "#4C72B0"), (REZAGADA, "#C44E52")]:
    axes[1].plot(suave.index, suave[ciudad] / 1000, color=color, linewidth=3.5)
    axes[1].annotate(f"{ciudad}  {crecimiento[ciudad]:+.0f} %",
                     xy=(suave.index[-1], suave[ciudad].iloc[-1] / 1000),
                     xytext=(8, 0), textcoords="offset points",
                     color=color, fontsize=14, fontweight="bold", va="center")
axes[1].set_title(f"{REZAGADA} es la segunda plaza del país y la que menos crece",
                  fontsize=17, loc="left", pad=14)
axes[1].set_ylabel("ventas mensuales\n(miles de dólares)", fontsize=13)
axes[1].tick_params(labelsize=12)
axes[1].set_xlim(suave.index.min(), suave.index.max() + pd.Timedelta(days=210))
axes[1].grid(axis="x", visible=False)
for lado in ["top", "right"]:
    axes[1].spines[lado].set_visible(False)

plt.tight_layout()
plt.show()

### Las siete diferencias, para que las copies

| # | exploración | defensa |
|---|---|---|
| 1 | **Título descriptivo**: «monto por mes y ciudad» | **Título con la conclusión**: «Guayaquil es la segunda plaza del país y la que menos crece» |
| 2 | Las seis series compiten entre sí | **Dos series a color, cuatro en gris**. El gris no es relleno: es contexto |
| 3 | Leyenda en una esquina, hay que ir y volver | **Etiqueta pegada a la línea**, con el número dentro |
| 4 | Tipografía de 6 y 7 puntos | **Tipografía de 12 a 17 puntos**. Si no se lee impreso al 50 %, no se lee proyectado |
| 5 | Datos crudos, mes a mes, con todo el ruido | **Media móvil de tres meses**: se ve la tendencia, que es de lo que se habla |
| 6 | El eje dice «monto» | El eje dice **«ventas mensuales (miles de dólares)»**: unidad explícita |
| 7 | Rejilla completa, marco de cuatro lados | **Sin rejilla vertical, sin marco**. Todo lo que no aporta, fuera |

📌 **La regla operativa: un gráfico de defensa, una frase.** Si necesitas dos frases para explicar tu
gráfico, son dos gráficos. Y si el panel tiene que preguntar «¿qué estoy mirando?», el gráfico ya
falló, por bonito que sea.

Prueba de fuego antes de proyectar: **imprímelo, aléjate tres pasos y léelo en voz alta.** Si no puedes
decir la conclusión sin acercarte, arréglalo.

## 2. Diez minutos con la conclusión en el primer minuto

La **pirámide de Minto** invierte el orden natural del trabajo. Tú llegaste a la recomendación al
final, después de limpiar, explorar y modelar. El panel la necesita **al principio**, porque cada minuto
que pasa sin saber a dónde vas lo gasta adivinando en lugar de escuchando.

La estructura es siempre la misma:

```
                   RECOMENDACIÓN
              (una frase, con su cifra)
                        │
        ┌───────────────┼───────────────┐
   ARGUMENTO 1     ARGUMENTO 2     ARGUMENTO 3
        │               │               │
   evidencia       evidencia       evidencia
   (un gráfico)    (una tabla)     (un número)
```

Diez minutos, siete bloques, y ninguno se puede saltar.

In [ ]:
PLAN = [
    ("Recomendación", 1.0,
     "Qué hay que hacer el lunes, cuánto vale al año y quién lo ejecuta. Sin preámbulo."),
    ("Contexto y pregunta", 1.0,
     "El negocio en dos frases y la pregunta que se contesta. Nada de historia de la empresa."),
    ("Argumento 1 · el hallazgo", 2.0,
     "El gráfico de defensa. Una idea, un gráfico, una frase."),
    ("Argumento 2 · el modelo", 2.0,
     "Qué predice, contra qué línea base y con qué error en unidades de negocio."),
    ("Argumento 3 · el dinero", 1.5,
     "Cuánto vale la recomendación al año, con los dos supuestos declarados en pantalla."),
    ("Limitaciones", 1.5,
     "Lo que el análisis NO puede decir, y qué haría falta para poder decirlo."),
    ("Cierre y próximo paso", 1.0,
     "Repetir la recomendación palabra por palabra y pedir la decisión concreta."),
]

plan = pd.DataFrame(PLAN, columns=["sección", "minutos", "qué se dice"])
plan["desde"] = plan["minutos"].cumsum() - plan["minutos"]
plan["hasta"] = plan["minutos"].cumsum()
plan["% del tiempo"] = plan["minutos"] / plan["minutos"].sum() * 100
plan = plan[["sección", "desde", "hasta", "minutos", "% del tiempo", "qué se dice"]]

print(f"Guion de {plan['minutos'].sum():.0f} minutos · {len(plan)} bloques\n")
print(plan.to_string(index=False, float_format=lambda v: f"{v:,.1f}", max_colwidth=64))
print(f"\nla recomendación se dice en el minuto {plan.loc[0, 'desde']:.1f} y se repite en el "
      f"{plan.loc[len(plan) - 1, 'desde']:.1f}")
print(f"tiempo dedicado a evidencia (argumentos 1, 2 y 3): "
      f"{plan.loc[2:4, 'minutos'].sum():.1f} min = {plan.loc[2:4, '% del tiempo'].sum():.0f} %")
print(f"tiempo dedicado a cómo lo hiciste: 0.0 min = 0 %  ← esto no es un olvido")

📌 **El 55 % del tiempo es evidencia y el 0 % es metodología.** Cómo limpiaste los datos, qué librería
usaste y en qué orden hiciste las cosas **no van en los diez minutos**: van en el cuaderno, y el panel
pregunta si le interesa. Los dos minutos que la mayoría de los grupos gasta contando el proceso son los
dos minutos que después les faltan para el dinero y las limitaciones.

Dos detalles del guion que parecen menores y no lo son:

- **La recomendación se dice dos veces**, al principio y al final, con las mismas palabras. No es
  redundancia: es lo único que el panel va a recordar mañana, y la repetición literal es lo que hace
  que la recuerden igual todos.
- **Las limitaciones ocupan minuto y medio, más que el dinero.** Un grupo que declara sus límites antes
  de que se los encuentren controla la conversación. Un grupo al que se los encuentran, la pierde.

El cronómetro no es opcional. Un guion de diez minutos sin ensayar dura catorce.

In [ ]:
import time


def temporizador(plan, segundos_por_minuto=60.0):
    '''Recorre el guion anunciando cada bloque.

    En el ensayo real se llama sin argumentos (60 s por minuto). Para probar que
    funciona sin esperar diez minutos, se baja segundos_por_minuto.
    '''
    inicio = time.perf_counter()
    registro = []
    for _, fila in plan.iterrows():
        t0 = time.perf_counter()
        print(f"[{fila['desde']:>4.1f}′ → {fila['hasta']:>4.1f}′]  {fila['sección']:<26s} "
              f"({fila['minutos']:.1f} min)")
        time.sleep(fila["minutos"] * segundos_por_minuto)
        registro.append({"sección": fila["sección"], "previsto (min)": fila["minutos"],
                         "real (s)": time.perf_counter() - t0})
    total = time.perf_counter() - inicio
    print(f"\n⏱  fin del ensayo · {total:.2f} s simulando "
          f"{plan['minutos'].sum():.0f} minutos reales")
    return pd.DataFrame(registro)


# Simulación rápida: 0,05 segundos por cada minuto de exposición.
registro = temporizador(plan, segundos_por_minuto=0.05)
print()
print(registro.to_string(index=False, float_format=lambda v: f"{v:,.3f}"))
print("\nPara el ensayo de verdad: temporizador(plan)  ← sin argumentos, un minuto es un minuto")

Cómo se usa en el ensayo, en tres reglas:

1. **Se ensaya de pie y en voz alta**, con el cronómetro corriendo y sin parar aunque te equivoques.
   Parar y repetir entrena a parar y repetir.
2. **Si un bloque se pasa, se recorta ese bloque**, no el siguiente. El error clásico es robarle tiempo
   a las limitaciones porque el hallazgo se alargó.
3. **Tres ensayos completos como mínimo.** El primero descubre que no cabe, el segundo arregla el
   guion, el tercero es el que se parece a la defensa.

## 3. El verificador de tu propio cuaderno

El panel no va a ejecutar tu cuaderno delante de ti, pero el docente sí, en una máquina limpia. Cinco
controles deciden si el cuaderno es entregable, y los cinco se pueden comprobar **sin abrirlo**.

La función siguiente lee cualquier archivo `.ipynb` y devuelve una lista de aprobado o pendiente.
Apúntala a tu propio archivo antes de subirlo.

In [ ]:
import json
import re

PISTAS_RUTA_LOCAL = ["/users/", "/home/", "c:\\", "c:/", "d:\\", "/desktop/", "/escritorio/",
                     "/descargas/", "/downloads/", "/documents/", "/documentos/"]
PISTAS_LINEA_BASE = ["dummyregressor", "dummyclassifier", "línea base", "linea base",
                     "línea de base", "baseline"]
PISTAS_BITACORA = ["bitácora de prompts", "bitacora de prompts", "bitácora de IA",
                   "registro de prompts"]
PISTAS_LIMITES = ["limitacion", "limitación", "limitaciones", "qué no puede decir",
                  "sesgo", "supuesto"]


def verificar_cuaderno(ruta):
    '''Cinco controles de entregabilidad sobre un archivo .ipynb.

    Devuelve un DataFrame con APROBADO o PENDIENTE y la evidencia de cada control.
    '''
    nb = json.loads(Path(ruta).read_text(encoding="utf-8"))
    celdas = nb["cells"]
    codigo = [c for c in celdas if c["cell_type"] == "code" and "".join(c["source"]).strip()]
    fuente = "\n".join("".join(c["source"]) for c in codigo).lower()
    texto = "\n".join("".join(c["source"]) for c in celdas
                      if c["cell_type"] == "markdown").lower()
    todo = fuente + "\n" + texto
    controles = []

    # 1 · ¿corre completo y en orden?
    ejec = [c.get("execution_count") for c in codigo]
    sin_ejecutar = sum(e is None for e in ejec)
    en_orden = ejec == list(range(1, len(ejec) + 1))
    controles.append((
        "Corre completo y de arriba abajo",
        en_orden and len(ejec) > 0,
        f"{len(ejec)} celdas de código · {sin_ejecutar} sin ejecutar · "
        f"numeración {'consecutiva 1…' + str(len(ejec)) if en_orden else 'desordenada: ' + str(ejec[:8])}"))

    # 2 · ¿los datos se leen de una fuente accesible?
    lecturas = re.findall(r"read_(csv|excel|parquet|json|table)", fuente)
    locales = [p for p in PISTAS_RUTA_LOCAL if p in fuente]
    tiene_url = "http://" in fuente or "https://" in fuente
    controles.append((
        "Los datos se leen de fuente accesible",
        len(lecturas) > 0 and not locales,
        f"{len(lecturas)} lectura(s) de archivo · "
        f"{'URL presente' if tiene_url else 'sin URL'} · "
        f"{'rutas locales detectadas: ' + ', '.join(locales) if locales else 'sin rutas locales'}"))

    # 3 · ¿hay línea base declarada?
    encontradas = [p for p in PISTAS_LINEA_BASE if p in todo]
    controles.append((
        "Línea base declarada",
        len(encontradas) > 0,
        f"señales: {', '.join(encontradas) if encontradas else 'ninguna'}"))

    # 4 · ¿existe la bitácora de prompts?
    bit = [p for p in PISTAS_BITACORA if p in todo]
    controles.append((
        "Bitácora de prompts presente",
        len(bit) > 0,
        f"señales: {', '.join(bit) if bit else 'ninguna'}"))

    # 5 · ¿están escritas las limitaciones? (y con cuerpo, no una línea suelta)
    celdas_limite = [t for t in ["".join(c["source"]) for c in celdas
                                 if c["cell_type"] == "markdown"]
                     if any(p in t.lower() for p in PISTAS_LIMITES)]
    palabras = sum(len(t.split()) for t in celdas_limite)
    controles.append((
        "Limitaciones escritas (mínimo 80 palabras)",
        palabras >= 80,
        f"{len(celdas_limite)} celda(s) de texto hablan de límites · {palabras} palabras"))

    tabla = pd.DataFrame(controles, columns=["control", "ok", "evidencia"])
    tabla["estado"] = np.where(tabla["ok"], "APROBADO", "PENDIENTE")
    return tabla[["control", "estado", "evidencia"]]


print("verificar_cuaderno() lista ✓ · 5 controles")

In [ ]:
import tempfile

CARPETA = Path(tempfile.mkdtemp())


def cuaderno_de_prueba(nombre, celdas):
    ruta = CARPETA / nombre
    ruta.write_text(json.dumps({
        "cells": celdas,
        "metadata": {"kernelspec": {"display_name": "Python 3", "name": "python3"}},
        "nbformat": 4, "nbformat_minor": 5}, ensure_ascii=False), encoding="utf-8")
    return ruta


def md(texto):
    return {"cell_type": "markdown", "metadata": {}, "source": [texto]}


def code(texto, n):
    return {"cell_type": "code", "execution_count": n, "metadata": {}, "outputs": [],
            "source": [texto]}


LIMITES_BIEN = (
    "## Limitaciones\n\nEl modelo se entrena con 30 meses de una sola empresa, así que el "
    "coeficiente no se puede trasladar a otro distribuidor sin volver a estimarlo. La relación "
    "entre inversión y ventas es de asociación y no de causa: el presupuesto se fija mirando las "
    "ventas esperadas, así que sirve para pronosticar pero no para decidir cuánto invertir. El "
    "conjunto de prueba tiene ocho meses, de modo que el error medido es inestable y puede moverse "
    "varios puntos con dos meses distintos. No incluimos el efecto de la competencia porque no "
    "tenemos el dato, y ese es el supuesto más frágil del análisis. El sesgo por ciudad no se "
    "revisó para las plazas con menos de cien clientes.")

listo = cuaderno_de_prueba("proyecto_grupo_listo.ipynb", [
    md("# Pronóstico de ventas de Comercial Andina\n\nRecomendación: mover el presupuesto."),
    code("import pandas as pd\n"
         "d = pd.read_csv('https://raw.githubusercontent.com/x/y/datos.csv')", 1),
    code("from sklearn.dummy import DummyRegressor\n"
         "base = DummyRegressor(strategy='mean').fit(X, y)", 2),
    code("modelo = LinearRegression().fit(X, y)\nprint(modelo.score(X, y))", 3),
    md("## Bitácora de prompts\n\n| prompt | qué verifiqué |\n|---|---|\n| ... | ... |"),
    md(LIMITES_BIEN),
])

flojo = cuaderno_de_prueba("proyecto_grupo_flojo.ipynb", [
    md("# Análisis de ventas\n\nPrimero cargamos los datos y luego exploramos."),
    code("import pandas as pd\n"
         "d = pd.read_csv('C:/Users/ana/Desktop/ventas_final_v3_ok.csv')", 3),
    code("d.groupby('ciudad')['monto'].sum()", 1),
    code("modelo.fit(X, y)\nprint('R2', modelo.score(X, y))", None),
    md("Limitaciones: faltan datos."),
])

for ruta in [listo, flojo]:
    resultado = verificar_cuaderno(ruta)
    aprobados = (resultado["estado"] == "APROBADO").sum()
    print(f"\n══ {ruta.name} · {aprobados} de {len(resultado)} controles aprobados ══\n")
    print(resultado.to_string(index=False, max_colwidth=72))

📌 **El cuaderno flojo aprueba 0 de 5 controles y ninguno de los cinco tiene que ver con el análisis.**
Los cinco son de oficio, se arreglan en media hora y son exactamente los que hacen que el trabajo de un
semestre no se pueda reproducir.

Qué significa cada pendiente en la práctica:

- **Numeración desordenada** (`[3, 1, None]`): el cuaderno se ejecutó a saltos. Nadie sabe si el
  resultado que se ve corresponde al código que está escrito. Se arregla con *Reiniciar y ejecutar
  todo* antes de guardar, siempre.
- **Ruta local** (`C:/Users/ana/Desktop/ventas_final_v3_ok.csv`): el cuaderno no corre en ninguna otra
  máquina del planeta. Los datos van por URL o en una carpeta relativa dentro del repositorio.
- **Sin línea base**: el `R²` que imprime no se puede interpretar. Es la regla de la semana 11 y es la
  primera pregunta del panel.
- **Sin bitácora de prompts**: incumple la política de IA del curso. No es un tecnicismo: es la parte
  de la nota que se pierde entera.
- **«Limitaciones: faltan datos.»**: tres palabras. El umbral del verificador son ochenta, y ochenta
  palabras siguen siendo poco.

Corre el verificador sobre **tu** archivo antes de subirlo. Si alguno sale pendiente, arréglalo hoy:
todos se arreglan hoy.

In [ ]:
# Apunta esto a tu propio cuaderno antes de entregarlo.
MI_CUADERNO = Path("lab_16_apellido.ipynb")     # ← cámbialo por el archivo de tu grupo

if MI_CUADERNO.exists():
    print(verificar_cuaderno(MI_CUADERNO).to_string(index=False, max_colwidth=72))
else:
    print(f"No encuentro '{MI_CUADERNO}'. Cambia MI_CUADERNO por la ruta de tu archivo.")
    print("En Colab: sube el .ipynb con el panel de archivos y pon aquí su nombre.\n")
    candidatos = sorted(Path(".").glob("*.ipynb"))
    print(f"Cuadernos que veo en esta carpeta ({len(candidatos)}): "
          f"{[c.name for c in candidatos[:8]]}")

## 4. El banco de preguntas del panel

Un panel hace entre seis y diez preguntas en diez minutos. Casi todas salen de esta lista, porque son
las que hace cualquiera que sepa leer un análisis. **La preparación no consiste en memorizar
respuestas, sino en saber en qué celda del cuaderno está la evidencia** y poder llegar a ella en
quince segundos.

In [ ]:
PREGUNTAS = [
    ("¿Contra qué comparaste? ¿Cuál es la línea base?",
     "Es la primera y la más letal. Sin ella ningún número significa nada.",
     "Celda con DummyRegressor o DummyClassifier y la tabla con las dos métricas al lado",
     "Semana 11"),
    ("¿Por qué elegiste esa métrica y no otra?",
     "Distingue al que copió del que decidió.",
     "La frase con la tolerancia del negocio escrita ANTES de ver los resultados",
     "Semana 12"),
    ("¿Cuánto vale esto al año?",
     "Si no hay cifra, la recomendación no se aprueba.",
     "El cálculo con sus dos supuestos visibles y la fuente de cada supuesto",
     "Semanas 12 y 14"),
    ("¿Qué pasa si el supuesto principal no se cumple?",
     "Mide si el grupo entendió su propio modelo.",
     "Análisis de sensibilidad: el mismo cálculo con el supuesto movido ±20 %",
     "Semana 13"),
    ("¿Esto es causa o correlación?",
     "La trampa favorita del panel. Casi siempre es correlación.",
     "La limitación escrita y el experimento que haría falta para probar la causa",
     "Semana 9"),
    ("¿Por qué descartaste esa variable?",
     "Comprueba si hubo criterio o si el modelo lo escribió el asistente.",
     "La tabla de VIF o el p-valor, con el umbral declarado",
     "Semana 13"),
    ("¿A quién perjudica tu modelo?",
     "Si no hay respuesta, el modelo no se despliega.",
     "La tabla de métricas por subgrupo y la disparidad con su cifra",
     "Semana 15"),
    ("¿Qué le dices al cliente al que el modelo clasificó mal?",
     "Separa el modelo del sistema que lo usa.",
     "La explicación local y el canal de apelación",
     "Semana 15"),
    ("¿Cuántos datos tuviste que tirar y por qué?",
     "Mide la honestidad de la limpieza.",
     "La bitácora de limpieza con el conteo antes y después y el dinero involucrado",
     "Semana 4"),
    ("¿Esto lo escribió una IA?",
     "La respuesta 'lo generó el asistente' no es una respuesta.",
     "La bitácora de prompts, con qué se verificó de cada respuesta",
     "Todas"),
    ("¿Qué harías distinto con más datos o más tiempo?",
     "Pregunta amable que distingue al que tiene criterio del que se quedó sin ideas.",
     "Tres líneas concretas, no 'más datos'. Qué dato, para responder qué pregunta",
     "Semana 16"),
    ("Si tuvieras que hacer una sola cosa el lunes, ¿cuál?",
     "El cierre habitual. Es la recomendación otra vez, y hay que decirla igual.",
     "La misma frase del minuto 1, palabra por palabra",
     "Semana 16"),
]

banco = pd.DataFrame(PREGUNTAS, columns=["pregunta del panel", "por qué la hacen",
                                         "evidencia que hay que tener abierta", "viene de"])
banco.index = np.arange(1, len(banco) + 1)
banco["¿la tengo?"] = "___"

print(f"{len(banco)} preguntas · llena la última columna con el número de celda de tu cuaderno\n")
print(banco.to_string(max_colwidth=52))
print(f"\nLas tres letales, por orden: {banco.loc[1, 'pregunta del panel']} · "
      f"{banco.loc[3, 'pregunta del panel']} · {banco.loc[7, 'pregunta del panel']}")

📌 **Las tres letales son la línea base, el dinero y a quién perjudica.** Si el grupo no puede
contestar esas tres con una celda abierta en pantalla, la defensa se cae aunque el análisis sea
correcto, porque el panel no puede distinguir un análisis correcto de uno afortunado.

Dos reglas de respuesta que valen más que el contenido:

- **«No lo sé» es una respuesta válida. «Creo que…» no.** Si no lo mediste, dilo, y di qué medirías. El
  panel penaliza la invención mucho más que la ignorancia declarada.
- **Contesta primero, justifica después.** «Sí, la campaña se paga: 1 017 dólares de retorno sobre 507
  clientes con los supuestos de la lámina 6» y luego, si preguntan, el detalle. No al revés.

## 5. La rúbrica, antes de que la use el panel

Se publica entera y se lee antes de entregar. Puntúate tú primero: los grupos que se autoevalúan con
honestidad suben, de media, más que los que se sorprenden el día de la defensa.

In [ ]:
RUBRICA = [
    ("Recomendación y conclusión", 25,
     "Una frase accionable con cifra, destinatario y plazo. Se dice en el primer minuto",
     "Recomendación clara pero sin cifra o sin responsable",
     "Se describe lo que se hizo y no qué hay que hacer"),
    ("Evidencia y método", 20,
     "Cada argumento tiene su evidencia y el método es reproducible y justificado",
     "La evidencia existe pero no se conecta con el argumento",
     "Gráficos sin conclusión o modelo sin justificar"),
    ("Línea base y honestidad de las métricas", 15,
     "Línea base declarada, error en unidades de negocio y tolerancia escrita antes",
     "Hay línea base pero la métrica no se traduce al negocio",
     "Solo R² o solo exactitud, sin comparación"),
    ("Cuaderno reproducible", 15,
     "Corre completo en máquina limpia, datos accesibles, celdas en orden",
     "Corre con ajustes menores de ruta",
     "No corre o depende de archivos locales"),
    ("Limitaciones, sesgos y ética", 10,
     "Límites propios, disparidad medida por subgrupo y qué haría falta para resolverlos",
     "Limitaciones genéricas del tipo 'faltan datos'",
     "No hay sección de limitaciones"),
    ("Defensa oral y respuesta al panel", 10,
     "Diez minutos cronometrados y respuestas directas con evidencia en pantalla",
     "Se pasa de tiempo o responde sin mostrar evidencia",
     "Cuenta el proceso en orden cronológico"),
    ("Bitácora de prompts y uso declarado de IA", 5,
     "Prompt final y qué se verificó de cada respuesta",
     "Bitácora incompleta",
     "No hay bitácora"),
]

rubrica = pd.DataFrame(RUBRICA, columns=["criterio", "peso %", "sobresaliente (100 %)",
                                         "aceptable (60 %)", "insuficiente (0 %)"])
rubrica["mi autoevaluación"] = "___"

assert rubrica["peso %"].sum() == 100, "los pesos tienen que sumar 100"
print(f"{len(rubrica)} criterios · pesos suman {rubrica['peso %'].sum()} ✓\n")
print(rubrica.to_string(index=False, max_colwidth=46))
print(f"\nLos tres criterios que más pesan suman "
      f"{rubrica.nlargest(3, 'peso %')['peso %'].sum()} % de la nota: "
      f"{', '.join(rubrica.nlargest(3, 'peso %')['criterio'])}")
print(f"Todo lo que se arregla con oficio y no con análisis "
      f"(cuaderno reproducible + bitácora) vale "
      f"{rubrica.loc[rubrica['criterio'].isin(['Cuaderno reproducible', 'Bitácora de prompts y uso declarado de IA']), 'peso %'].sum()} %")

📌 **El 20 % de la nota —cuaderno reproducible más bitácora— no depende del análisis, solo del
oficio.** Es la parte más barata de conseguir y la que más grupos regalan. El verificador de la sección
3 cubre el 15 %; la bitácora, el 5 % restante.

Y al otro lado: **la recomendación vale 25 %, más que el método.** No es un desequilibrio: es el
mensaje del curso entero. Un análisis impecable que termina en «los datos muestran una correlación
interesante» no vale más que uno decente que termina en «mover el presupuesto de prensa a radio libera
1 200 dólares al mes sin perder ventas esperadas».

## 6. La evaluación 360: qué aprendiste y cómo funcionó el equipo

El curso cierra con la parte que nadie prepara y que todos recuerdan: **cómo trabajó el grupo**. Vale
el 5 % de la nota del curso y se entrega por separado, pero su utilidad real no es la nota: es que
obliga a poner por escrito lo que se estuvo diciendo en los pasillos durante cuatro meses.

Se llena con nombres, no con «el equipo», y con hechos, no con adjetivos.

In [ ]:
DIMENSIONES = [
    ("Contribución técnica", "¿Escribió código o análisis que quedó en la entrega final?",
     "una celda concreta que hizo esa persona"),
    ("Cumplimiento", "¿Entregó lo suyo a tiempo, sin que hubiera que perseguirlo?",
     "una fecha en la que entregó algo comprometido"),
    ("Criterio", "¿Cuestionó una decisión del grupo con un argumento y un número?",
     "la decisión que cambió por lo que dijo"),
    ("Comunicación", "¿Se puede entender lo que escribe sin pedirle explicaciones?",
     "un texto suyo que se usó tal cual"),
    ("Colaboración", "¿Ayudó a alguien del equipo cuando se atascó?",
     "el momento concreto"),
]

evaluacion_360 = pd.DataFrame(DIMENSIONES,
                              columns=["dimensión", "pregunta", "evidencia que hay que citar"])
evaluacion_360["puntaje 1-5"] = "___"
evaluacion_360["evidencia concreta"] = "___"

CIERRE = [
    "¿Cuál fue la decisión de análisis más difícil del semestre y por qué la tomaron así?",
    "¿Qué harían distinto si empezaran hoy el mismo proyecto?",
    "¿Qué error cometieron y en qué semana lo detectaron?",
    "¿Qué parte del trabajo le encargaron a la IA y qué tuvieron que corregirle?",
    "¿Qué se lleva cada uno que no sabía en la semana 1?",
]

print("Formulario 360 · una copia por cada integrante del equipo, incluido uno mismo\n")
print(evaluacion_360.to_string(index=False, max_colwidth=58))
print(f"\n\nCinco preguntas de cierre del equipo (media página cada una, entre todos):\n")
for i, pregunta in enumerate(CIERRE, 1):
    print(f"  {i}. {pregunta}")
print(f"\ntotal a entregar: {len(evaluacion_360)} dimensiones × integrantes del equipo "
      f"+ {len(CIERRE)} respuestas conjuntas")

### 🌶️ Ejercicio 1 — Guiado

Toma **tu** gráfico principal, el que va a ir en la lámina del argumento 1, y conviértelo en gráfico de
defensa aplicando las siete diferencias de la sección 1. Entrega los dos, antes y después, en la misma
figura, y debajo una lista de cuáles de las siete aplicaste y cuáles no tenían sentido en tu caso.
Después haz la prueba: escribe el título con la conclusión y comprueba que se puede leer solo.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: fig, axes = plt.subplots(1, 2, figsize=(15, 4.6)) y repite el mismo dato en los dos
# Pista 2: el gris del contexto es "#D9D9D9"; los protagonistas, "#4C72B0" y "#C44E52"
# Pista 3: ax.annotate(texto, xy=(x_final, y_final), xytext=(8, 0), textcoords="offset points")
#          pone la etiqueta pegada a la línea, que es lo que sustituye a la leyenda
# Pista 4: si tu conclusión necesita dos frases, divídelo en dos gráficos

### 🔥 Desafío

**Escribe la recomendación de una página.** Es la entrega que más pesa y la que se deja para el final.
Estructura obligatoria, una sola página:

1. **La recomendación**, una frase con cifra, destinatario y plazo.
2. **Tres argumentos**, un párrafo cada uno, cada uno con **un número** y de dónde sale.
3. **Lo que vale al año**, con los dos supuestos escritos entre paréntesis.
4. **Tres limitaciones**, específicas de tu análisis, no genéricas.
5. **El próximo paso**, una acción con nombre y fecha.

Después pásala por dos filtros: (a) que un gerente que no sepa estadística pueda leerla en voz alta sin
tropezar, y (b) que **no aparezcan** las palabras «modelo», «significativo», «coeficiente», «R²» ni
«correlación». Si alguna es imprescindible, tradúcela.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: escríbela en una celda de texto y usa el código solo para calcular las cifras que cites,
#          y para comprobar que no se cuela ninguna palabra prohibida:
#          PROHIBIDAS = ["modelo", "significativo", "coeficiente", "r²", "correlación"]
# Pista 2: cuenta las palabras. Una página son entre 400 y 500. Si pasas de 600, sobra un argumento
# Pista 3: la frase de la recomendación es la MISMA del minuto 1 del guion. Cópiala, no la reescribas

### 🎯 Reto en clase (15 min)

En equipos y contra reloj: **la defensa de noventa segundos**. Cada grupo tiene noventa segundos —no
diez minutos— para dar recomendación, un argumento y el dinero. Un cronómetro visible corta a los
noventa segundos exactos, a mitad de frase si hace falta.

Después, **el panel rota**: cada grupo hace las tres preguntas letales de la sección 4 a otro grupo, y
las respuestas se puntúan con dos criterios únicamente: si se contestó en los primeros diez segundos y
si se mostró evidencia. Gana el equipo que no tuvo que decir «déjame buscarlo».

Si tu defensa no cabe en noventa segundos, el problema no es el tiempo: es que todavía no sabes cuál es
tu conclusión.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: temporizador(pd.DataFrame([...]), segundos_por_minuto=60) con un plan de 1,5 minutos
# Pista 2: el plan de noventa segundos son tres bloques: recomendación (20 s), argumento (50 s),
#          dinero (20 s). Escríbelo como DataFrame con las columnas sección, desde, hasta, minutos
# Pista 3: apunta cuántas de las tres preguntas letales pudiste contestar sin buscar. Ese número
#          es tu nivel de preparación real

## La trampa de hoy

⚠️ **Contar el proceso en orden cronológico.** Es la trampa más difícil de evitar porque es la más
natural: hiciste el trabajo en ese orden y te parece que así se entiende. El panel no quiere saber qué
hiciste primero; quiere saber qué hay que hacer el lunes.

Los dos guiones, con el mismo contenido y la misma duración, medidos por el único número que importa:
**en qué minuto se entera el panel de la recomendación.**

In [ ]:
CRONOLOGICO = [
    ("La empresa y el sector", 1.5), ("Cómo conseguimos los datos", 1.0),
    ("Limpieza de datos", 1.5), ("Exploración inicial", 1.5),
    ("Los modelos que probamos", 2.0), ("Resultados", 1.5),
    ("Conclusión y recomendación", 1.0),
]

crono = pd.DataFrame(CRONOLOGICO, columns=["sección", "minutos"])
crono["desde"] = crono["minutos"].cumsum() - crono["minutos"]


def minuto_de_la_recomendacion(guion, clave="ecomendaci"):
    fila = guion[guion["sección"].str.contains(clave)]
    return float(fila["desde"].iloc[0])


m_crono = minuto_de_la_recomendacion(crono)
m_minto = minuto_de_la_recomendacion(plan)
DURACION = plan["minutos"].sum()

comparacion = pd.DataFrame({
    "guion cronológico (el natural)": [
        m_crono, m_crono / DURACION * 100, DURACION - m_crono,
        crono.loc[crono["sección"].isin(["Cómo conseguimos los datos", "Limpieza de datos",
                                         "Exploración inicial"]), "minutos"].sum()],
    "pirámide de Minto (el correcto)": [
        m_minto, m_minto / DURACION * 100, DURACION - m_minto, 0.0],
}, index=["minuto en que aparece la recomendación",
          "% de la charla que el panel escucha sin saber a dónde vas",
          "minutos que el panel tiene la recomendación en la cabeza",
          "minutos dedicados a contar el proceso"])

print(f"Los dos guiones duran lo mismo: {DURACION:.0f} y {crono['minutos'].sum():.0f} minutos\n")
print(comparacion.to_string(float_format=lambda v: f"{v:,.1f}"))
print(f"\nEl número equivocado : «lo conté todo, y en el orden en que pasó»")
print(f"El número correcto   : la recomendación llega en el minuto {m_crono:.1f} contra "
      f"{m_minto:.1f}")
print(f"                       el panel pasa {m_crono / DURACION * 100:.0f} % del tiempo "
      f"adivinando, contra {m_minto / DURACION * 100:.0f} %")
print(f"                       y solo la retiene {DURACION - m_crono:.1f} minutos "
      f"contra {DURACION - m_minto:.1f}")

📌 **Con el guion cronológico la recomendación llega en el minuto 9,0 de 10: el panel pasa el 90 % del
tiempo sin saber a dónde vas y solo tiene la conclusión en la cabeza durante 1 minuto.** Con la
pirámide de Minto llega en el minuto 0,0 y la retiene los 10.

Y hay un daño peor que el tiempo. En el guion cronológico, las preguntas empiezan **antes** de que
llegues a la recomendación —porque el panel se impacienta— así que las contestas fuera de contexto, te
quedas sin minutos y terminas leyendo la conclusión a toda prisa mientras alguien mira el reloj. Es el
final más común de las defensas malas y no tiene nada que ver con la calidad del análisis.

La regla, y con esto cierra el curso: **empieza por el final.** La conclusión primero, la evidencia
después, el proceso solo si preguntan. Lo mismo vale para el correo que vas a escribir el lunes en tu
trabajo, para la reunión del martes y para el informe del viernes.

## Entregable

El proyecto final se entrega completo:

- **Recomendación de una página** con la estructura del desafío: una frase con cifra, tres argumentos
  con su número, el valor anual con los dos supuestos, tres limitaciones específicas y el próximo paso
  con nombre y fecha.
- **Cuaderno reproducible** que pasa los cinco controles de `verificar_cuaderno()`. Ejecuta el
  verificador sobre tu propio archivo y **deja la salida visible** en `lab_16_apellido.ipynb`.
- **Bitácora de prompts completa**, con el prompt final de cada entrega y una línea sobre qué
  verificaste de la respuesta.
- **Defensa de diez minutos** ensayada con el temporizador, con la recomendación en el primer minuto y
  repetida en el último.
- **Banco de preguntas** con la última columna llena: para cada una de las doce, el número de celda de
  tu cuaderno que la contesta.
- **Autoevaluación con la rúbrica** antes de la defensa, con la nota que crees que te corresponde en
  cada criterio y una línea de por qué.
- **Evaluación 360 entre pares**, que se entrega por separado.

## Para tu equipo

- **Ensayen tres veces completo y de pie.** El primer ensayo descubre que no cabe en diez minutos, el
  segundo arregla el guion y el tercero es el que se parece a la defensa. Sin los tres, el que se
  parece a la defensa es el día de la defensa.
- **Repártanse las preguntas, no las secciones.** Quien presenta el modelo no tiene por qué contestar
  sobre el dinero. Decidan antes quién contesta cada una de las doce preguntas del banco y díganlo en
  voz alta una vez.
- **Ejecuten el cuaderno en una máquina que no sea la suya** —otro portátil, o Colab desde cero— el día
  anterior. Es donde aparecen las rutas locales, las librerías que solo tenía uno y las celdas que
  dependían de otra que borraron.